# SparseVol3D â€” Google Colab Setup

**One-click setup for running SparseVol3D on Google Colab (free GPU)**

### Before running:
1. Go to **Runtime â†’ Change runtime type â†’ T4 GPU**
2. Then click **Runtime â†’ Run all**

---
**What this notebook does:**
- Installs all dependencies
- Clones the SparseVol3D repo from GitHub
- Downloads KiTS19 data (or generates synthetic data for a quick test)
- Trains the model
- Evaluates and plots results

## Step 1 â€” Check GPU

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected.')
    print('Go to Runtime > Change runtime type > T4 GPU')

## Step 2 â€” Install dependencies & clone repo

In [ ]:
# Install required packages
!pip install nibabel tqdm scikit-image scipy -q

# Clone or update the SparseVol3D repo
import os
if not os.path.exists('SparseVol3D'):
    !git clone https://github.com/NatalieCarlebach1/SparseVol3D.git
else:
    !cd SparseVol3D && git pull   # always pull latest changes

os.chdir('SparseVol3D')
print('Working directory:', os.getcwd())

import sys
sys.path.insert(0, os.getcwd())

## Step 3 â€” Data

**Option A** (recommended for real results): download KiTS19 (~50 GB, ~1 hour)

**Option B** (quick test, ~10 seconds): use synthetic data

Set `USE_SYNTHETIC = True` for a quick smoke test, or `False` to download the real dataset.

In [ ]:
USE_SYNTHETIC = True   # <-- change to False to download real KiTS19 data

DATA_DIR = 'data/kits19/data'
os.makedirs(DATA_DIR, exist_ok=True)

if USE_SYNTHETIC:
    print('Generating 15 synthetic cases...')
    !python scripts/make_synthetic_data.py --n_cases 15 --data_dir {DATA_DIR}
else:
    print('Downloading KiTS19 (~50 GB)...')
    if not os.path.exists('../kits19'):
        !git clone https://github.com/neheller/kits19.git ../kits19
    !cd ../kits19 && pip install -r requirements.txt -q
    !cd ../kits19 && python starter_code/get_imaging.py
    if not os.path.exists(DATA_DIR):
        os.symlink('../kits19/data', DATA_DIR)

import glob
cases = sorted(glob.glob(os.path.join(DATA_DIR, 'case_*')))
print(f'\nReady: {len(cases)} cases in {DATA_DIR}')

In [ ]:
import json

!python scripts/prepare_splits.py --data_dir {DATA_DIR}

with open('outputs/splits.json') as f:
    splits = json.load(f)
print(f"Train: {len(splits['train'])}  Val: {len(splits['val'])}  Test: {len(splits['test'])}")

## Step 4 â€” Configure & Train

Change `LABEL_STRIDE` and `LAMBDA_VIC` to run different experiments.

In [ ]:
LABEL_STRIDE   = 5      # 1=dense (100%), 5=every 5th (20%), 10=every 10th (10%)
LAMBDA_VIC     = 0.1    # 0.0 = no VIC loss (ablation), 0.1 = SparseVol3D
USE_COORD_MLP  = False  # True = add NeRF-inspired CoordMLP (set False for baseline)
EPOCHS         = 100
suffix         = "_coord" if USE_COORD_MLP else ""
OUTPUT_DIR     = f"outputs/colab_s{LABEL_STRIDE}_vic{LAMBDA_VIC}{suffix}"

print(f"label_stride  = {LABEL_STRIDE}  ({100 // LABEL_STRIDE}% annotation)")
print(f"lambda_vic    = {LAMBDA_VIC}")
print(f"use_coord_mlp = {USE_COORD_MLP}")

In [ ]:
coord_flag = "--use_coord_mlp" if USE_COORD_MLP else ""
!python train.py \n    --data_dir     {DATA_DIR} \n    --output_dir   {OUTPUT_DIR} \n    --label_stride {LABEL_STRIDE} \n    --lambda_vic   {LAMBDA_VIC} \n    --epochs       {EPOCHS} \n    {coord_flag}

## Step 5 â€” Evaluate

In [ ]:
# Make sure variables are set even if this cell is run standalone
try:
    OUTPUT_DIR
    DATA_DIR
except NameError:
    LABEL_STRIDE = 5
    LAMBDA_VIC   = 0.1
    OUTPUT_DIR   = f'outputs/colab_s{LABEL_STRIDE}_vic{LAMBDA_VIC}'
    DATA_DIR     = 'data/kits19/data'

import os
ckpt_path = f'{OUTPUT_DIR}/best_model.pt'
if not os.path.exists(ckpt_path):
    print(f'ERROR: checkpoint not found at {ckpt_path}')
    print('Make sure Step 4 (training) completed successfully.')
else:
    # --splits_file defaults to outputs/splits.json in evaluate.py
    !python evaluate.py \
        --checkpoint {OUTPUT_DIR}/best_model.pt \
        --data_dir   {DATA_DIR} \
        --split      val

## Step 6 â€” Plot Results

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt

with open(f'{OUTPUT_DIR}/train_log.json') as f:
    log = json.load(f)

epochs_with_dice = [r for r in log if 'kidney_dice' in r]
ep = [r['epoch']       for r in epochs_with_dice]
kd = [r['kidney_dice'] for r in epochs_with_dice]
td = [r['tumor_dice']  for r in epochs_with_dice]
ls = [r['train_loss']  for r in log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(range(1, len(ls)+1), ls, color='steelblue', lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training Loss'); ax1.grid(alpha=0.3)

ax2.plot(ep, kd, label='Kidney', color='royalblue', lw=2)
ax2.plot(ep, td, label='Tumor',  color='tomato',    lw=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Dice')
ax2.set_title(f'Validation Dice (stride={LABEL_STRIDE}, VIC={LAMBDA_VIC})')
ax2.legend(); ax2.grid(alpha=0.3); ax2.set_ylim(0, 1)

plt.tight_layout(); plt.show()

print(f'Best Kidney Dice : {max(kd):.4f}')
print(f'Best Tumor  Dice : {max(td):.4f}')
print(f'Best Mean   Dice : {(max(kd)+max(td))/2:.4f}')

### Visual Predictions â€” random val cases

In [ ]:
import random, torch, numpy as np, nibabel as nib, matplotlib.pyplot as plt, os, sys
sys.path.insert(0, os.getcwd())
from models import UNet3D

# Load checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt   = torch.load(f"{OUTPUT_DIR}/best_model.pt", map_location="cpu", weights_only=False)
cfg_d  = ckpt["config"]
model  = UNet3D(
    in_channels=1,
    num_classes=cfg_d.get("num_classes", 3),
    base_channels=cfg_d.get("base_channels", 32),
    use_coord_mlp=cfg_d.get("use_coord_mlp", False),
    coord_features=cfg_d.get("coord_features", 16),
    coord_freq_bands=cfg_d.get("coord_freq_bands", 6),
).to(device)
model.load_state_dict(ckpt["model"])
model.eval()
patch_size = tuple(cfg_d.get("patch_size", [64, 128, 128]))
print(f"Model loaded  (CoordMLP={cfg_d.get('use_coord_mlp', False)})")

def preprocess(img):
    return ((np.clip(img, -200, 300) + 200.0) / 500.0).astype(np.float32)

def predict_patch(img_pp):
    D, H, W    = img_pp.shape
    pd, ph, pw = patch_size
    d0 = max(0, (D - pd) // 2)
    h0 = max(0, (H - ph) // 2)
    w0 = max(0, (W - pw) // 2)
    patch = img_pp[d0:d0+pd, h0:h0+ph, w0:w0+pw]
    pads  = [(0, max(0, t - s)) for s, t in zip(patch.shape, patch_size)]
    patch = np.pad(patch, pads)
    with torch.no_grad():
        t    = torch.from_numpy(patch[None, None]).float().to(device)
        pred = model(t).argmax(dim=1).cpu().numpy()[0]
    return patch, pred, d0, h0, w0

# Pick 3 random val cases
val_cases = splits["val"]
chosen    = random.sample(val_cases, min(3, len(val_cases)))
n_slices  = 3

fig, axes = plt.subplots(len(chosen) * 2, n_slices, figsize=(n_slices * 4, len(chosen) * 5))
fig.suptitle("Predictions on random val cases
(blue=kidney, red=tumor)", fontsize=13)
if len(chosen) == 1:
    axes = axes.reshape(2, n_slices)

for row_pair, case_id in enumerate(chosen):
    case_dir  = os.path.join(DATA_DIR, f"case_{case_id:05d}")
    img_raw   = nib.load(f"{case_dir}/imaging.nii.gz").get_fdata(dtype="float32")
    seg_raw   = nib.load(f"{case_dir}/segmentation.nii.gz").get_fdata().astype(int)
    img_pp    = preprocess(img_raw)
    patch, pred, d0, h0, w0 = predict_patch(img_pp)
    seg_patch = seg_raw[d0:d0+patch_size[0], h0:h0+patch_size[1], w0:w0+patch_size[2]]
    pads      = [(0, max(0, t - s)) for s, t in zip(seg_patch.shape, patch_size)]
    seg_patch = np.pad(seg_patch, pads)
    D         = patch.shape[0]
    show_z    = [D // 4, D // 2, 3 * D // 4]
    for col, z in enumerate(show_z):
        ct = patch[z] * 500 - 200
        for row, (data, title) in enumerate([
            (seg_patch, f"GT  case_{case_id:05d}  z={z}"),
            (pred,      f"Pred  z={z}"),
        ]):
            ax = axes[row_pair * 2 + row, col]
            ax.imshow(ct, cmap="gray", vmin=-200, vmax=300)
            ax.imshow(np.ma.masked_where(data[z]!=1, data[z]), cmap="Blues", alpha=0.5, vmin=0, vmax=2)
            ax.imshow(np.ma.masked_where(data[z]!=2, data[z]), cmap="Reds",  alpha=0.6, vmin=0, vmax=2)
            ax.set_title(title, fontsize=8)
            ax.axis("off")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/visual_predictions.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR}/visual_predictions.png")

## Step 7 â€” Save to Google Drive (optional)

Mount your Drive to keep the checkpoint after the Colab session ends.

In [ ]:
SAVE_TO_DRIVE = False   # set True to save checkpoint to Google Drive

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    dest = f'/content/drive/MyDrive/SparseVol3D/{os.path.basename(OUTPUT_DIR)}'
    shutil.copytree(OUTPUT_DIR, dest, dirs_exist_ok=True)
    print(f'Saved to {dest}')

## Step 8 — Full Ablation Experiments (optional)

Runs all label_stride x lambda_vic combinations + CoordMLP variants.
Takes ~8-12 hours on a T4 GPU for 100 epochs.

In [ ]:
# Run all ablation experiments
# --coord_mlp also runs NeRF CoordMLP variants
!python run_experiments.py \n    --data_dir {DATA_DIR} \n    --epochs   100 \n    --coord_mlp